# llm_board

> The task board in words (Phase 10): rules, observations with the open puzzle and the claims, parsing (anything unusable becomes a counted "wait"), and voluntary claims to the claim board or messages.

In [ ]:
#| default_exp llm_board

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from explRL.board_agents import BoardLead, ClaimBoard
from explRL.harness import MessageBus
from explRL.llm_env import BOARD_SCHEMA


class TextBoardEnv:
    "The task board in words: the open subtasks, the agent's open puzzle, claims; parsed actions and voluntary claims."
    task, timeUnit, SCHEMA, unit = 'board', 'turn', BOARD_SCHEMA, 'points'

    def __init__(self, board):
        self.world, self.shown, self._heard = board, {}, {}

    def rules(self):
        b = self.world
        vals = ', '.join(f'{v:g}' for v in sorted(set(b.values)))
        return (f"You are one of a team of {b.nAgents} solvers working on a shared board of subtasks. The board always "
                f"shows {2 * b.nAgents + 2} open subtasks, named T followed by a number. Each subtask is worth "
                f"{vals} points (shown on the board) and stays open for {b.life:g} turns after it appears; when it "
                "expires or is solved, a new one takes its place at once.\n\n"
                "Each subtask is a puzzle: a short Python function f of an integer x, built from +, *, ^ (bitwise "
                "exclusive or) and % (remainder), and a target y. A correct answer is an integer x in the stated "
                "range with f(x) == y; there may be several. Work it out carefully: you cannot run code.\n\n"
                "Turns: each of your turns is one action and takes one turn of time. 'open' a subtask (target = its "
                "name) to see its puzzle from your next turn on; opening costs that turn. 'submit' an answer (an "
                "integer in answer) to the subtask you have open; you may submit again after a wrong answer, as long "
                "as the subtask is open. 'wait' does nothing.\n\n"
                f"Scoring: only the first correct answer to a subtask scores its value; a correct answer to a subtask "
                f"that was already solved or has expired scores nothing, and every wrong answer costs {b.penalty:g} "
                "points. Two solvers working on the same subtask is wasted work, because only one of them can score.\n\n"
                "Claims: after any turn you may post a claim about one subtask: 'working' (you are on it) or "
                "'solved' (it is done); use claimKind 'none' for no claim. Depending on the team's set-up, claims go to "
                "a shared board every teammate sees, to two random teammates as a message, or nowhere; a coordinator, "
                "if there is one, suggests a subtask to each solver. Claims are voluntary and may be wrong, and you are "
                "free to ignore any suggestion.\n\n"
                "Worked example: def f(x): return ((x * 3) + 4) % 10, find x with 0 <= x < 50 and f(x) == 7. Try "
                "x = 1: 1 * 3 = 3, 3 + 4 = 7, 7 % 10 = 7, so x = 1 is correct (so are 11, 21, 31 and 41). For ^, write "
                "both numbers in binary and set each bit where exactly one of them has it: 5 ^ 3 = 101 ^ 011 = 110 = "
                "6, and 12 ^ 7 = 1100 ^ 0111 = 1011 = 11. Applying ^ k twice gives back the start, and % keeps the "
                "remainder after division (17 % 5 = 2), so working backwards from y is often faster than guessing.\n\n"
                "What you see each turn: the turn number, what your last action earned, the open subtasks with their "
                "values and the turns they have left, your open subtask's puzzle (if any), and whatever your team's "
                "set-up shows: the latest claims on the board, the messages you received, or the coordinator's "
                "suggestion. Below that come your own past turns (or your latest notes).\n\n"
                "Reply with JSON only: action is 'open', 'submit' or 'wait'; target is the subtask name for 'open' "
                "(empty otherwise); answer is your integer for 'submit' (0 otherwise); claimTask and claimKind give "
                "your claim; notes is a short note to yourself (what you know, what you plan), shown back to you later.")

    def observe(self, i, t, channel=None):
        b = self.world
        b.advance(t)
        o = b.observe(i)
        lines = [f"Turn {t:g}. Your last action earned {o['lastReward']:.2f}.",
                 'Open subtasks (value, turns left): ' + ', '.join(f'T{k} {v:g} ({e - t:g} left)' for k, v, e in o['open'])]
        k = o['opened']
        if k is None:
            lines.append('You have no subtask open.')
        else:
            state = 'open' if k in b.open else 'closed (solved or expired)'
            lines.append(f'Your open subtask: T{k}, {state}.\n' + b.tasks[k].puzzle.text())
        shown = set()
        if isinstance(channel, ClaimBoard):
            cl = channel.view(i, t, b.open)
            shown = {c[0] for c in cl}
            lines.append('Claims: ' + ('; '.join(f'T{k} {kind} (agent {a}, {age:g} turns ago)' for k, kind, a, age in cl)
                                       or 'none'))
        elif isinstance(channel, MessageBus):
            msgs = channel.take(i)
            shown = {j for _, j, _, _ in msgs}
            self._heard[i] = [f"T{j} {'working' if v == 1 else 'solved'}" for _, j, v, _ in msgs]
            lines.append('Messages: ' + ('; '.join(f"agent {s} says T{j} {'working' if v == 1 else 'solved'}"
                                                    for s, j, v, _ in msgs) or 'none'))
        elif isinstance(channel, BoardLead):
            j = channel.assign(i, b)
            shown = {j}
            lines.append(f'The coordinator suggests: T{j}.')
        self.shown[i] = shown
        return '\n'.join(lines)

    def _task(self, s):
        s = str(s).strip().upper()
        return int(s[1:]) if s[:1] == 'T' and s[1:].isdigit() else None

    def parse(self, parsed, i):
        "-> (action, claim (task, kind) or None, valid); anything unusable is 'wait', invalid."
        b = self.world
        if not isinstance(parsed, dict):
            return ('wait', None), None, False
        claim, ct = None, self._task(parsed.get('claimTask', ''))
        if parsed.get('claimKind') in ('working', 'solved') and ct in b.tasks:
            claim = (ct, parsed['claimKind'])
        a = parsed.get('action')
        if a == 'wait':
            return ('wait', None), claim, True
        if a == 'open':
            k = self._task(parsed.get('target', ''))
            return (('open', k), claim, True) if k in b.open else (('wait', None), claim, False)
        ans = parsed.get('answer')
        if a == 'submit' and b.opened[i] is not None and isinstance(ans, int) and not isinstance(ans, bool):
            return ('submit', ans), claim, True
        return ('wait', None), claim, False

    def post(self, i, claim, t, channel, info):
        if claim is None or channel is None:
            return False
        k, kind = claim
        if isinstance(channel, ClaimBoard):
            channel.post(i, k, kind, t)
        elif isinstance(channel, MessageBus):
            return bool(channel.due(i, t) and channel.send(self.world, i, k, 1.0 if kind == 'working' else 2.0, t))
        else:
            return False                       # the coordinator assigns from the board itself
        return True

    def explored(self, i, action, info):
        "Opening a subtask no teammate's claim or suggestion pointed to this turn."
        return info['kind'] == 'travel' and action[1] not in self.shown.get(i, ())

    def summary(self, i, t, action, reward, claim, info):
        verb = {'opened': f'opened T{info["task"]}', 'solved': f'solved T{info["task"]}',
                'wrong': f'answered {action[1]} to T{info["task"]}: wrong', 'late': f'answered T{info["task"]} too late',
                'wait': 'waited', 'invalid': 'made an invalid move'}[info['result']]
        s = f'turn {t:g}: {verb}, earned {reward:.2f}'
        if claim:
            s += f'; you claimed T{claim[0]} {claim[1]}'
        if self._heard.get(i):
            s += '; messages: ' + ', '.join(self._heard[i])
        return s + '.'

## Tests

In [ ]:
import numpy as np
from explRL.llm_board import *
from explRL.board_agents import BoardLead, ClaimBoard
from explRL.harness import MessageBus
from explRL.multi import spawnRngs
from explRL.taskboard import TaskBoardEnv

b = TaskBoardEnv(0, 2, difficulty=1)
env = TextBoardEnv(b)
assert len(env.rules()) / 5 > 512, len(env.rules())
obs = env.observe(0, 0.0)
assert all(f'T{k} ' in obs for k in b.open) and 'no subtask open' in obs
k = b.open[0]
P = lambda **kw: dict(dict(action='wait', target='', answer=0, claimTask='', claimKind='none', notes=''), **kw)
assert env.parse(P(action='open', target=f't{k}'), 0) == (('open', k), None, True)
assert env.parse(P(action='open', target='T999'), 0) == (('wait', None), None, False)
assert env.parse(P(action='submit', answer=3), 0) == (('wait', None), None, False)      # nothing open yet
assert env.parse(None, 0) == (('wait', None), None, False)
b.apply(0, ('open', k), 0.0)
obs = env.observe(0, 1.0)
assert f'Your open subtask: T{k}, open.' in obs and b.tasks[k].puzzle.text() in obs
assert env.parse(P(action='submit', answer=b.tasks[k].puzzle.answer, claimTask=f'T{k}', claimKind='solved'), 0) == \
    (('submit', b.tasks[k].puzzle.answer), (k, 'solved'), True)
assert env.parse(P(action='submit', answer=True), 0)[2] is False

# claims reach the claim board, two teammates, or nobody (the lead assigns from the board)
b = TaskBoardEnv(1, 3, difficulty=1); env = TextBoardEnv(b); k = b.open[2]
cb = ClaimBoard()
assert env.post(0, (k, 'working'), 0.0, cb, {}) and f'T{k} working (agent 0' in env.observe(1, 1.0, cb)
assert env.shown[1] == {k} and env.explored(1, ('open', b.open[0]), dict(kind='travel'))
assert not env.explored(1, ('open', k), dict(kind='travel'))
mb = MessageBus(3, 2, 1.0, np.inf, spawnRngs(4, 3))
assert env.post(0, (k, 'solved'), 0.0, mb, {})
assert f'agent 0 says T{k} solved' in env.observe(1, 1.0, mb) and f'T{k} solved' in env.observe(2, 1.0, mb)
lead = BoardLead(3)
assert 'The coordinator suggests: T' in env.observe(0, 1.0, lead) and not env.post(0, (k, 'working'), 1.0, lead, {})
r, _, _, info = b.apply(0, ('open', k), 1.0)
assert env.summary(0, 1.0, ('open', k), r, None, info) == f'turn 1: opened T{k}, earned 0.00.'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()